In [10]:
import pandas as pd

# Load dataset from local path
dataset_path = "raja_ravi_varma_dataset.csv"  # keep this file in the same folder as the notebook
df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print(df.head())


Dataset loaded successfully!
   id                   title  year        category  \
0   1              Shakuntala  1870    Mythological   
1   2       Lady in Moonlight  1890        Portrait   
2   3  Damayanti and the Swan  1878    Mythological   
3   4  Maharani of Travancore  1880  Royal Portrait   
4   5               Saraswati  1896    Mythological   

                                           image_url  
0  https://upload.wikimedia.org/wikipedia/commons...  
1  https://upload.wikimedia.org/wikipedia/commons...  
2  https://upload.wikimedia.org/wikipedia/commons...  
3  https://upload.wikimedia.org/wikipedia/commons...  
4  https://upload.wikimedia.org/wikipedia/commons...  


In [13]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu


In [21]:
# 1) Setup: imports and device
import os
import math
import random
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, utils

# Reproducibility
seed = 42
random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


# ART-IMAGE-GENERATOR — Corrected Notebook

This cleaned notebook uses PyTorch to train a simple DCGAN on portrait images (64×64). Epochs reduced for faster runs. Adjust `DATA_DIR` and `EPOCHS` as needed.

In [19]:
# 2) Config - change these if needed
DATA_DIR = '../input/art-portraits/Portraits'  # path to folder with images (ImageFolder expects subfolders; if images are directly inside, set root accordingly)
IMAGE_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 10  # reduced as requested
LATENT_DIM = 100
LR = 2e-4
BETA1 = 0.5  # Adam beta1
OUTPUT_DIR = '/mnt/data/art_gan_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Data dir exists?', os.path.exists(DATA_DIR))

Data dir exists? False


In [ ]:
# 3) Dataset and DataLoader
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # scale to [-1,1]
])

# Try to load using ImageFolder. If the dataset is a flat directory, use datasets.ImageFolder with a fake class folder.
if os.path.isdir(DATA_DIR):
    dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
    # If the root contains images directly and no subfolders, ImageFolder will fail. Try an alternative:
    if len(dataset.classes) == 0:
        raise RuntimeError('No classes found by ImageFolder. Make sure your images are in subfolders or change DATA_DIR.')
else:
    raise RuntimeError(f'DATA_DIR not found: {DATA_DIR}')

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print('Number of images:', len(dataset))

In [ ]:
# 4) Utility: weights init and fixed noise for sampling
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)  # for sampling

In [ ]:
# 5) Define Generator and Discriminator (DCGAN-style)
class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            # input Z goes into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. (ngf) x 32 x 32
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # state size. (nc) x 64 x 64
        )

    def forward(self, input):
        return self.main(input)


class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            # input is (nc) x 64 x 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input).view(-1, 1).squeeze(1)

In [ ]:
# 6) Instantiate models, optimizers, and loss
netG = Generator(nz=LATENT_DIM).to(device)
netD = Discriminator().to(device)

netG.apply(weights_init)
netD.apply(weights_init)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))

print(netG)
print(netD)

In [ ]:
# 7) Training loop (clean, with progress bars)
img_list = []
G_losses = []
D_losses = []
iters = 0

print('Starting Training Loop...')
for epoch in range(1, EPOCHS + 1):
    pbar = tqdm(dataloader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for i, (data, _) in enumerate(pbar):
        ############################
        # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
        ###########################
        netD.zero_grad()
        # train with real
        real_cpu = data.to(device)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), 1., dtype=torch.float, device=device)
        output = netD(real_cpu)
        errD_real = criterion(output, label)
        errD_real.backward()
        D_x = output.mean().item()

        # train with fake
        noise = torch.randn(b_size, LATENT_DIM, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(0.)
        output = netD(fake.detach())
        errD_fake = criterion(output, label)
        errD_fake.backward()
        D_G_z1 = output.mean().item()
        errD = errD_real + errD_fake
        optimizerD.step()

        ############################
        # (2) Update G network: maximize log(D(G(z)))
        ###########################
        netG.zero_grad()
        label.fill_(1.)  # fake labels are real for generator cost
        output = netD(fake)
        errG = criterion(output, label)
        errG.backward()
        D_G_z2 = output.mean().item()
        optimizerG.step()

        # Save losses for plotting later
        G_losses.append(errG.item())
        D_losses.append(errD.item())

        if (i % 100) == 0:
            with torch.no_grad():
                fake_grid = netG(fixed_noise).detach().cpu()
                img_list.append(utils.make_grid(fake_grid, padding=2, normalize=True))

        iters += 1
        pbar.set_postfix({'errD': errD.item(), 'errG': errG.item()})

    # save checkpoint after each epoch
    torch.save(netG.state_dict(), os.path.join(OUTPUT_DIR, f'netG_epoch{epoch}.pth'))
    torch.save(netD.state_dict(), os.path.join(OUTPUT_DIR, f'netD_epoch{epoch}.pth'))

print('Training finished.')

In [ ]:

# 8) Plot losses and show a sample image grid
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))
plt.plot(G_losses, label='G_loss')
plt.plot(D_losses, label='D_loss')
plt.legend()
plt.title('Training losses')
plt.show()

# Show last generated images grid if available
if img_list:
    # img_list contains torchvision grids (C x H x W)
    last_grid = img_list[-1]
    npimg = last_grid.permute(1,2,0).numpy()  # H x W x C
    plt.figure(figsize=(8,8))
    plt.axis('off')
    plt.title('Generated images (last saved grid)')
    plt.imshow((npimg * 0.5) + 0.5)  # denormalize from [-1,1] to [0,1]
    plt.show()
else:
    print('No generated images to display.')


In [ ]:
# 9) Save a few sample images to OUTPUT_DIR
from torchvision.utils import save_image
with torch.no_grad():
    sample = netG(fixed_noise).detach().cpu()
    save_image((sample + 1) / 2, os.path.join(OUTPUT_DIR, 'samples_epoch_final.png'), nrow=8)
print('Saved samples to', OUTPUT_DIR)